In [2]:
import os
import urllib3

from tqdm import tqdm

import pandas as pd
import polars as pl

In [3]:
from io import BytesIO

pd_read_csv_kwargs = {'sep': None, 'encoding': 'latin-1', 'encoding_errors': 'ignore', 'on_bad_lines': 'skip', 'engine': 'python'}

def csv_base(rsc_url, response, **kwargs):
    # sometimes the data are encoded, sometimes not
    # and we do not want to start reading zip files here
    assert not rsc_url.endswith('.zip')
    assert 'DOCTYPE' not in response.data[:100].decode('latin-1')
    try:
        return pd.read_csv(response.data)
    except:            
        return pd.read_csv(BytesIO(response.data), **pd_read_csv_kwargs, **kwargs)

In [100]:
http = urllib3.PoolManager(headers={
            'User-Agent': 'Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:136.0) Gecko/20100101 Firefox/136.0'
        })

In [4]:
tables_path     = '../data/datasets/CAN/tables/tables_from10000_to15000'
metadata_path   = '../data/datasets/CAN/metadata/metadata_from10000_to15000'

In [56]:
tables_ids = list(set(map(lambda f: f.removesuffix('.parquet').split('_')[0], os.listdir(tables_path))))
len(tables_ids)

1600

In [57]:
tables_ids[:10]

['4b168d19-8abb-474c-b7ac-5dde1073df03',
 '6ea882a2-d661-4a37-9f79-832d02b053ab',
 'e5afeb2d-00cf-49ca-b757-c78cf48f463f',
 'b9676a41-8a89-4209-8b38-df7acf035f8c',
 'a034b41f-9fc6-4f1f-86fd-1726f6994c89',
 '3bb4e82c-a4f8-4662-8cd7-cf538bf3aba0',
 'dcc081b3-4168-4015-9442-d411f6f99b8a',
 '8ce165c6-28e7-4409-a0a5-516d0c62a9c7',
 '7ebcccaa-e64e-4ddb-8c28-99aa870ee890',
 '5992bb93-0a74-410f-8fda-c41f04ae61b2']

In [ ]:
import jsonlines

metadata = {}

with jsonlines.open(metadata_path) as fr:
    for line in tqdm(fr.iter()):
        for resource in line['resources']:
            if resource['id'] in set(tables_ids):
                metadata[resource['id']] = resource['url']            
len(metadata)

2842it [00:01, 2282.54it/s]


1600

In [5]:
errors = 0
for table in tqdm(os.listdir(tables_path)):
    try:
        pl.read_parquet(f'{tables_path}/{table}')
    except:
        errors += 1
errors

  0%|          | 0/3689 [00:00<?, ?it/s]

100%|██████████| 3689/3689 [00:09<00:00, 383.75it/s]


0

In [116]:
i = 13
tables_ids[i]

'b87b04ad-498d-4756-bdc1-f678fc58e82f'

In [117]:
df = pd.read_parquet(f"{tables_path}/{tables_ids[i]}.parquet")
df

,Sample Plan Number / Numéro du plan d'échantillonnage,Sampling Plan - Description / Plan déchantillonnage - Description,System ID / ID système,Inspection Sample Number / Numéro dinspection de léchantillon,Job Laboratory Number / Tâche  numéro de laboratoire,Date Sampled - Date (YYYY/MM/DD) / Date d'échantillonnage - Date (AAAA/MM/JJ),Product Type / Type de produit,Country of Origin / Pays dorigine,Domestic and/or Import // Canadienne et/ou Importation,Analysis Type / Type danalyse,Method Number / Numéro de méthode,Analysis Assessment / Évaluation de l'analyse
0,2020_SB3091,"Imported scallops (pre-packaged, frozen)/ Péto...",2020FSY-0000057721-4,V2020CCSH00984,STH-FD-2020-VI-0920,11/22/2020,Frozen raw scallops/ Pétoncles crus congelés,China/ Chine,Import/ Importation,Norovirus (GII),CFSAN/ACIA-CRNVA-05,Satisfactory / Satisfaisante
1,2020_SB3091,"Imported scallops (pre-packaged, frozen)/ Péto...",2020FSY-0000057721-4,V2020CCSH00984,STH-FD-2020-VI-0920,11/22/2020,Frozen raw scallops/ Pétoncles crus congelés,China/ Chine,Import/ Importation,Hepatitis A / Hépatite A,CFSAN/ACIA-CRNVA-05,Satisfactory / Satisfaisante
2,2020_SB3091,"Imported scallops (pre-packaged, frozen)/ Péto...",2020FSY-0000057721-4,V2020CCSH00984,STH-FD-2020-VI-0920,11/22/2020,Frozen raw scallops/ Pétoncles crus congelés,China/ Chine,Import/ Importation,Norovirus (GI),CFSAN/ACIA-CRNVA-05,Satisfactory / Satisfaisante
3,2020_SB3091,"Imported scallops (pre-packaged, frozen)/ Péto...",2021FSY-0000011100-4,V2020CCSH00987,STH-FD-2020-VI-1368,2/22/2021,Frozen raw scallops/ Pétoncles crus congelés,China/ Chine,Import/ Importation,Norovirus (GI),CFSAN/ACIA-CRNVA-05,Satisfactory / Satisfaisante
4,2020_SB3091,"Imported scallops (pre-packaged, frozen)/ Péto...",2021FSY-0000011100-4,V2020CCSH00987,STH-FD-2020-VI-1368,2/22/2021,Frozen raw scallops/ Pétoncles crus congelés,China/ Chine,Import/ Importation,Norovirus (GII),CFSAN/ACIA-CRNVA-05,Satisfactory / Satisfaisante
...,...,...,...,...,...,...,...,...,...,...,...,...
682,2021_SB3091,"Imported scallops (pre-packaged, frozen)/ Péto...",2021FSY-0000049714-4,V2021WCSH01865,STH-FD-2021-VI-0820,9/6/2021,Frozen raw scallops/ Pétoncles crus congelés,China/ Chine,Import/ Importation,Norovirus (GI),CFSAN/ACIA-CRNVA-05,Satisfactory / Satisfaisante
683,2021_SB3091,"Imported scallops (pre-packaged, frozen)/ Péto...",2021FSY-0000049714-4,V2021WCSH01865,STH-FD-2021-VI-0820,9/6/2021,Frozen raw scallops/ Pétoncles crus congelés,China/ Chine,Import/ Importation,Norovirus (GII),CFSAN/ACIA-CRNVA-05,Satisfactory / Satisfaisante
684,2021_SB3091,"Imported scallops (pre-packaged, frozen)/ Péto...",2022FSY-0000005758-4,V2021WCSH01906,STH-FD-2021-VI-1603,1/31/2022,Frozen raw scallops/ Pétoncles crus congelés,China/ Chine,Import/ Importation,Norovirus (GII),CFSAN/ACIA-CRNVA-05,Satisfactory / Satisfaisante
685,2021_SB3091,"Imported scallops (pre-packaged, frozen)/ Péto...",2022FSY-0000005758-4,V2021WCSH01906,STH-FD-2021-VI-1603,1/31/2022,Frozen raw scallops/ Pétoncles crus congelés,China/ Chine,Import/ Importation,Norovirus (GI),CFSAN/ACIA-CRNVA-05,Satisfactory / Satisfaisante


In [115]:
rsc_url = metadata[tables_ids[i]]
response = http.request('GET', rsc_url)
rsc_url

'https://www150.statcan.gc.ca/n1/tbl/csv/27100360-eng.zip'

In [102]:
response.data

b'\xef\xbb\xbfProgram name,Service provider,Local,Toll free\r\n"<a href=""https://www.kidsthrive.ca/"">Algoma Preschool Speech and Language</a>",THRIVE Child Development Centre,"<a href=""tel:705-759-1131"">705-759-1131</a>","<a href=""tel:1-855-759-1131"">1-855-759-1131</a>"\r\n"<a href=""http://www.lansdownecentre.ca/"">Brant Haldimand Norfolk Preschool Speech and Language</a>",Lansdowne Children\xe2\x80\x99s Centre,"<a href=""tel:519-753-3153"">519-753-3153 <abbr title=""extension"">ext.</abbr> 249</a> (Brant) | <a href=""tel:519-753-3153"">519-753-3153 <abbr title=""extension"">ext.</abbr> 247</a> (Haldimand Norfolk)","<abbr title=""Not available"">N/A</abbr>"\r\n"<a href=""http://www.childrenstreatment-ck.com/"">Chatham-Kent Preschool Speech and Language</a>",Children\xe2\x80\x99s Treatment Centre of Chatham Kent,"<a href=""tel:519-354-0520"">519-354-0520</a>","<abbr title=""Not available"">N/A</abbr>"\r\n"<a href=""https://www.porcupinehu.on.ca/en/your-family/speech/"">Preschool 

In [103]:
df = csv_base(rsc_url, response)
df

,ï»¿Program name,Service provider,Local,Toll free
0,"<a href=""https://www.kidsthrive.ca/"">Algoma Pr...",THRIVE Child Development Centre,"<a href=""tel:705-759-1131"">705-759-1131</a>","<a href=""tel:1-855-759-1131"">1-855-759-1131</a>"
1,"<a href=""http://www.lansdownecentre.ca/"">Brant...",Lansdowne Childrenâs Centre,"<a href=""tel:519-753-3153"">519-753-3153 <abbr ...","<abbr title=""Not available"">N/A</abbr>"
2,"<a href=""http://www.childrenstreatment-ck.com/...",Childrenâs Treatment Centre of Chatham Kent,"<a href=""tel:519-354-0520"">519-354-0520</a>","<abbr title=""Not available"">N/A</abbr>"
3,"<a href=""https://www.porcupinehu.on.ca/en/your...",Cochrane Temiskaming Childrenâs Treatment Ce...,"<a href=""tel:705-264-4700"">705-264-4700</a>","<a href=""tel:1-800-461-1818"">1-800-461-1818</a..."
4,"<a href=""https://www.erinoakkids.ca/home.aspx""...",ErinoakKids Centre for Treatment and Development,"<a href=""tel:905-855-2690"">905-855-2690</a>, t...","<a href=""tel:1-877-374-6625"">1-877-374-6625</a..."
5,"<a href=""https://grandviewkids.ca/"">Durham Pre...",Grandview Childrenâs Centre,"<a href=""tel:905-728-1673"">905-728-1673 <abbr ...","<a href=""tel:1-800-304-6180"">1-800-304-6180</a>"
6,"<a href=""http://www.eohu.ca/"">Eastern Ontario ...",Eastern Ontario Health Unit,"<abbr title=""Not available"">N/A</abbr>","<a href=""tel:1-800-267-7120"">1-800-267-7120</a>"
7,"<a href=""http://www.connectwithus.ca/"">Essex P...",Connections Early Years Family Centre,"<a href=""tel:519-252-0636"">519-252-0636</a>","<abbr title=""Not available"">N/A</abbr>"
8,"<a href=""http://www.tvcc.on.ca/"">Grey Bruce Pr...",Thames Valley Childrenâs Centre,"<abbr title=""Not available"">N/A</abbr>","<a href=""tel:1-866-590-8822"">1-866-590-8822</a>"
9,"<a href=""https://www.kidsability.ca/"">Guelph-W...",KidsAbility Centre for Child Development,"<a href=""tel:519-886-8886"">519-886-8886 <abbr ...","<a href=""tel:1-888-372-2259"">1-888-372-2259</a>"
